# Multimodal Mental-Health Screening — Colab GPU runner

Runtime > Change runtime type > **GPU (T4)**. Then Runtime > Run all.

Clones the repo, pulls the dataset from the GitHub release, runs audit +
LightGBM baseline + the full training ablation on GPU.

In [ ]:
# 1) GPU check
import torch, subprocess
print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv'))
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 2) Clone repo
%cd /content
!rm -rf h4h_hackathon
!git clone --depth 1 https://github.com/hemish22/h4h_hackathon.git
%cd /content/h4h_hackathon

In [ ]:
# 3) Dependencies (Colab already has torch/numpy/pandas/sklearn/matplotlib/seaborn/Pillow)
!pip install -q librosa lightgbm shap scikit-image tabulate pyyaml soundfile

In [ ]:
# 4) Dataset: download the release tarball and extract into ./dataset
import os
if not os.path.exists('dataset/mental_health_multimodal.csv'):
    !wget -q -O dataset.tgz https://github.com/hemish22/h4h_hackathon/releases/download/data-v1/dataset.tgz
    # extract quietly; drop macOS AppleDouble (._*) sidecar files
    !tar --warning=no-unknown-keyword -xzf dataset.tgz 2>/dev/null
    !find dataset -name '._*' -delete
    !rm -f dataset.tgz
!ls dataset && python -c "import glob,os;print('wavs', len(glob.glob('dataset/Audios/Actor_*/*.wav')));print('imgs', sum(len(os.listdir(f'dataset/Extracted_images/{d}')) for d in os.listdir('dataset/Extracted_images')))"

In [ ]:
# 5) Sanity: smoke test + audit (manifests/splits are committed, no extraction needed)
!python -m tests.test_smoke
!python -m src.audit

In [ ]:
# 6) LightGBM tabular baseline (ablation row 1)
!python -m src.baseline

In [ ]:
# 7) Train the ablation. num_workers=2 is fine on Colab (Linux).
!python -m src.train --config configs/random.yaml   --workers 2   # row 7 (random pairing)
!python -m src.train --config configs/matched.yaml  --workers 2   # row 8 (matched pairing)
!python -m src.train --config configs/final.yaml    --workers 2   # row 10 (final)

In [ ]:
# 8) Collect results
import json, glob
for f in sorted(glob.glob('results/*/metrics.json')):
    d = json.load(open(f)); c = d.get('classification') or {}
    print(f.split('/')[-2], 'acc', round(c.get('accuracy',0),3), 'macroF1', round(c.get('macro_f1',0),3), 'qwk', round(c.get('qwk_EXTRA',0),3))

In [ ]:
# 9) (optional) zip trained models + results to download
!zip -qr artifacts_results.zip artifacts results figures reports && echo 'download artifacts_results.zip from the Files panel'